<a href="https://colab.research.google.com/github/amarjithanand/Student-Productivity-Tracker/blob/main/StudentProductivityTracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub scikit-learn joblib pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "sehaj1104/student-productivity-and-digital-distraction-dataset",
    "student_productivity_distraction_dataset_20000.csv"
)

df.head()

/tmp/ipython-input-3954027878.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 1.50M/1.50M [00:00<00:00, 6.55MB/s]


,student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score
0,1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78
1,2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.90,48.99
2,3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.60
3,4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87
4,5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.90


In [ ]:
FEATURES = [
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "stress_level",
    "focus_score",
    "attendance_percentage",
    "exercise_minutes",
    "coffee_intake_mg",
    "assignments_completed",
    "final_grade"
]

TARGET = "productivity_score"

df = df[FEATURES + [TARGET]]

In [ ]:
for col in FEATURES:
    df[col] = df[col].fillna(df[col].median())

df[TARGET] = df[TARGET].fillna(df[TARGET].median())

/tmp/ipython-input-1121167916.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(df[col].median())
/tmp/ipython-input-1121167916.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[TARGET] = df[TARGET].fillna(df[TARGET].median())


In [ ]:
bins = [-np.inf, 40, 70, np.inf]
labels = ["Distracted", "Neutral", "Productive"]
df["productivity_label"] = pd.cut(df[TARGET], bins=bins, labels=labels)

/tmp/ipython-input-1605152872.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["productivity_label"] = pd.cut(df[TARGET], bins=bins, labels=labels)


In [ ]:
X = df[FEATURES].values
y_reg = df[TARGET].values
y_clf = df["productivity_label"].values

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Regression
reg_baseline = LinearRegression()
reg_baseline.fit(X_train_scaled, y_reg_train)

reg_model = RandomForestRegressor(n_estimators=200, random_state=42)
reg_model.fit(X_train_scaled, y_reg_train)

# Classification
clf_baseline = LogisticRegression(max_iter=1000)
clf_baseline.fit(X_train_scaled, y_clf_train)

clf_model = RandomForestClassifier(n_estimators=200, random_state=42)
clf_model.fit(X_train_scaled, y_clf_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
# Regression evaluation
pred_reg = reg_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_reg_test, pred_reg))
r2 = r2_score(y_reg_test, pred_reg)
print("Regression RMSE:", rmse)
print("Regression R2:", r2)

# Classification evaluation
pred_clf = clf_model.predict(X_test_scaled)
print(classification_report(y_clf_test, pred_clf))

Regression RMSE: 2.4372545977002997
Regression R2: 0.9772921361338256
              precision    recall  f1-score   support

  Distracted       0.94      0.87      0.91      1108
     Neutral       0.90      0.97      0.93      2428
  Productive       0.98      0.72      0.83       464

    accuracy                           0.92      4000
   macro avg       0.94      0.86      0.89      4000
weighted avg       0.92      0.92      0.91      4000



In [ ]:
joblib.dump(reg_model, "regressor.pkl")
joblib.dump(clf_model, "classifier.pkl")
joblib.dump(scaler, "scaler.pkl")

with open("preprocess.json", "w") as f:
    json.dump({"feature_order": FEATURES}, f, indent=2)

print("Saved: regressor.pkl, classifier.pkl, scaler.pkl, preprocess.json")

Saved: regressor.pkl, classifier.pkl, scaler.pkl, preprocess.json


In [ ]:
from google.colab import files
files.download("regressor.pkl")
files.download("classifier.pkl")
files.download("scaler.pkl")
files.download("preprocess.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>